# H&M M5_D — M2+M3+M4 / seed49 단일 학습

**확정 적응형 M2 표현 + 중심화 N/V M3 그래프 + 고객 내 평균1 M4 손실 가중치**를 결합합니다.

- 새 학습은 M5_D 하나만 합니다. 기존 M1·M2·M3·M4 seed49 저장 결과는 재학습·재평가 없이 재사용합니다.
- 세 개입을 같은 forward·추천손실·하나의 optimizer로 공동 학습합니다. 기존 모델 checkpoint로 초기화하거나 동결하지 않습니다.
- train 2018-09-20~2020-09-15 / test 2020-09-16~22. **고정300epoch 마지막 점에서만 평가**합니다. validation·holdout·조기종료 없음.
- 기존 CLV 입력창365일, ID64·2층·batch131072·lr0.0005·ID L2 0.001·공유표현 L2 1e-6·uniform 음성1개·MIN_ITEM_INTER=1을 유지합니다. 학습 구매쌍은 정답·후보에서 제외합니다.
- 모델 수식·계수·epoch를 변경하지 않습니다. 이번 결과는 이미 관찰한 test의 seed49 기술적 비교이며, 10시드 평균·유의성·CLV 배정 귀속·독립 확증이 아닙니다.

현재 실험 ZIP 저장 후 빈 **L4 GPU**에서 위부터 실행하세요. 다른 진행 중인 런타임을 변경하지 마세요.
중단 시 같은 노트북·경로로 재실행하면 저장 지점에서 재개합니다. 완료 결과는 재학습·재평가하지 않습니다.
별도 M5_D 출력 폴더를 사용하므로 M5_A 및 기존 실험을 덮어쓰지 않습니다. M5_A/B/C 완료를 요구하지 않습니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, subprocess, sys, torch
if not Path('/content/drive/MyDrive').is_dir():
    drive.mount('/content/drive')
assert torch.cuda.is_available(), 'L4 GPU 런타임을 연결하세요.'
GPU = torch.cuda.get_device_name(0)
assert 'L4' in GPU, f'현재 GPU는 {GPU}입니다. 요청한 L4로 변경 후 실행하세요.'
SOURCE_COMMIT = 'e25bd6bd26e72c8e350aed7d41c9c043f4e13a75'
REPO = Path('/content/clv-hm-m5-d-s49-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', SOURCE_COMMIT], check=True)
sys.path.insert(0, str(REPO))
print('GPU:', GPU, '| 고정 코드:', SOURCE_COMMIT)

## 1. 기존 참조 확인·입력 준비 (학습 없음)
M2 원본과 M4 seed49 원본을 읽습니다. M4 원본에는 M1·M3 참조도 들어 있습니다.
원본 SHA256·seed·분할·학습조건·M2 특징·M3 그래프 진단·M4 구매행 가중치 해시를 대조하며 불일치하면 학습 전에 중단합니다.
참조가 없다고 해당 모형을 새로 학습하거나 순열 대조군을 추가하지 않습니다.

Drive 폴더의 보고서가 바뀌었다면 보관 원본 ZIP의 Drive 경로를 아래 두 인수에 지정하세요:
`clv_m2_adaptive_hm_seed49_lastweek_results.zip`, `clv_hm_m4c_lastweek_seed49_results.zip`.
첫 실행 이후에는 같은 참조·출력 경로를 유지하세요.

In [ ]:
import clv_hm_m5_d_adaptive_lastweek_seed49 as screen
cfg = screen.configure(
    # m2_reference='/content/drive/MyDrive/.../clv_m2_adaptive_hm_seed49_lastweek_results.zip',
    # m4_reference='/content/drive/MyDrive/.../clv_hm_m4c_lastweek_seed49_results.zip',
)
print('M2 참조:', cfg.m2_reference)
print('M1/M3/M4 참조:', cfg.m4_reference)
print('새 결과:', cfg.out_dir)
prepared = screen.prepare(cfg)
print('재사용:', [screen.LABELS[m] for m in prepared['references']['absolute'].model_id])
print('새 학습: M5_D(M2+M3+M4) seed49 하나, 300epoch / M4 가중 BPR')

## 2. M5_D 한 모형만 학습
학습 중 loss만 표시하고 test는 마지막300epoch에서 한 번만 평가합니다.
학습 중 다른 셀을 실행하거나 런타임을 바꾸지 마세요.

In [ ]:
result = screen.run(prepared)
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

## 3. 같은 seed49 결과 비교
M1·M2·M3·M4는 저장된 참조값이며 M5_D만 새 학습 결과입니다. **1시드 결과로 10시드 평균이 아닙니다.**
핵심12지표 외 전체142지표·CLV 세그먼트·노출·작동 진단도 CSV에 저장됩니다. 상승·하락 모두 확인하세요.
아래 표는 네 단독모형과 비교합니다. 기존 M5_A/B/C와의 직접 비교는 각 원본 결과가 확보된 뒤 동일조건으로 합쳐 분석합니다.
준비 셀 이후 이 셀은 저장 결과만 읽으며 학습·test 평가를 추가하지 않습니다.

In [ ]:
from IPython.display import display
result = screen.report(prepared)
metrics = list(screen.CORE)
display(result['absolute'].set_index('model')[metrics].T.reindex(
    columns=['M1', 'M2', 'M3', 'M4', 'M5_D']))
if result['complete']:
    display(result['comparison'].pivot(index='metric', columns='reference',
        values='relative_change_pct').reindex(metrics)[['M1', 'M2', 'M3', 'M4']])
    print('M5_D의 각 참조 대비 상대변화(%). 전체142지표는 CSV에 저장됩니다.')
else:
    print('M5_D 미완료: 위 표는 참조값만 표시합니다. 새 학습·평가하지 않습니다.')
print('seed49 1시드 결과 — 10시드 평균/유의성/CLV 귀속 주장 없음')
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))

In [ ]:
from google.colab import files
output = screen.export_zip(prepared, '/content/hm_m5_d_adaptive_lastweek_seed49_results.zip')
files.download(output)